In [ ]:
# MNIST Classification with Laplace Approximation

This notebook demonstrates multi-class classification on the MNIST dataset using Bayesian neural networks with Laplace approximation. 

## Key Features:
- **Dataset**: MNIST handwritten digits (0-9)
- **Model**: 3-layer MLP with ReLU activations
- **Bayesian Inference**: Laplace approximation for uncertainty quantification
- **Training**: Mini-batch gradient descent with Adam optimizer
- **Evaluation**: Accuracy metrics and uncertainty analysis

## Notebook Structure:
1. Data loading and preprocessing
2. Neural network architecture definition
3. Training with mini-batches
4. Laplace approximation fitting
5. Model evaluation and uncertainty analysis
6. Prediction visualization and comparison
7. Comprehensive performance analysis


In [18]:
import Pkg
Pkg.activate("../../../")
Pkg.instantiate()

  Activating project at `~/Desktop/LaplaceRedux.jl`


In [19]:
using LaplaceRedux.Data
using Flux
using MLDatasets
using Random
Random.seed!(42)

# Load MNIST dataset
train_x, train_y = MLDatasets.MNIST(split=:train)[:]
test_x, test_y = MLDatasets.MNIST(split=:test)[:]

# Reshape and normalize data
train_x = reshape(Float32.(train_x), 28*28, :) ./ 255.0f0
test_x = reshape(Float32.(test_x), 28*28, :) ./ 255.0f0

# Take a subset for faster training (first 5000 samples)
n_samples = 5000
train_x = train_x[:, 1:n_samples]
train_y = train_y[1:n_samples]

# Convert to vectors for LaplaceRedux
x = [train_x[:, i] for i in 1:size(train_x, 2)]
y = train_y

# Prepare data for training
X = train_x
y_train = Flux.onehotbatch(y, 0:9)  # MNIST has 10 classes (0-9)

10×5000 OneHotMatrix(::Vector{UInt32}) with eltype Bool:
 ⋅  1  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  …  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  1  ⋅  ⋅  ⋅  ⋅  ⋅
 ⋅  ⋅  ⋅  1  ⋅  ⋅  1  ⋅  1  ⋅  ⋅  ⋅  ⋅     ⋅  ⋅  ⋅  ⋅  ⋅  1  ⋅  ⋅  ⋅  ⋅  1  ⋅
 ⋅  ⋅  ⋅  ⋅  ⋅  1  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅     ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  1  ⋅  1
 ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  1  ⋅  ⋅  1  ⋅  1     1  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  1  ⋅  ⋅  ⋅
 ⋅  ⋅  1  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  1  ⋅  ⋅  ⋅     ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅
 1  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  1  ⋅  …  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅
 ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅     ⋅  ⋅  1  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅
 ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅     ⋅  ⋅  ⋅  1  ⋅  ⋅  ⋅  1  ⋅  ⋅  ⋅  ⋅
 ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅     ⋅  ⋅  ⋅  ⋅  1  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅
 ⋅  ⋅  ⋅  ⋅  1  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅     ⋅  1  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅

In [20]:
data = zip(x, y_train)
n_hidden = 50  # Increase hidden units for MNIST complexity
D = size(X, 1)  # 784 for MNIST (28*28)
out_dim = 10  # MNIST has 10 classes
nn = Chain(
    Dense(D, n_hidden, relu),
    Dense(n_hidden, n_hidden, relu),
    Dense(n_hidden, out_dim)
)  
loss(x, y) = Flux.Losses.logitcrossentropy(nn(x), y)

loss (generic function with 1 method)

In [21]:
using Flux.Optimise: update!, Adam
using Statistics
opt = Adam(0.001)  # Learning rate for MNIST
epochs = 50  # Fewer epochs for faster training
show_every = 10

for epoch = 1:epochs
    loss_sum = 0
    
    # Mini-batch training for better performance
    for batch in Flux.DataLoader((X, y_train), batchsize=64, shuffle=true)
        x_batch, y_batch = batch
        gs = gradient(Flux.params(nn)) do
            loss(x_batch, y_batch)
        end
        update!(opt, Flux.params(nn), gs)
        loss_sum += loss(x_batch, y_batch)
    end
    
    if epoch % show_every == 0
        println("Epoch ", epoch)
        println("Average Loss: ", loss_sum / length(Flux.DataLoader((X, y_train), batchsize=64)))
        
        # Calculate accuracy
        predictions = nn(X)
        predicted_classes = Flux.onecold(predictions, 0:9)
        accuracy = mean(predicted_classes .== y)
        println("Accuracy: ", round(accuracy * 100, digits=2), "%")
    end
end

Epoch 10
Average Loss: 0.7505669
Accuracy: 76.92%
Epoch 20
Average Loss: 0.5464629
Accuracy: 84.52%
Epoch 30
Average Loss: 0.4373271
Accuracy: 87.8%
Epoch 40
Average Loss: 0.372672
Accuracy: 89.68%
Epoch 50
Average Loss: 0.32199576
Accuracy: 90.68%


In [ ]:
using LaplaceRedux
la = Laplace(nn; likelihood=:classification)
fit!(la, data)
optimize_prior!(la; verbosity=1, n_steps=100)  # Fewer steps for faster optimization

In [ ]:
# Evaluate model on test set
test_predictions = predict(la, test_x[:, 1:1000])  # Use first 1000 test samples
test_predictions_matrix = reduce(hcat, test_predictions)
test_predicted_classes = Flux.onecold(test_predictions_matrix, 0:9)
test_accuracy = mean(test_predicted_classes .== test_y[1:1000])

println("Test Accuracy: ", round(test_accuracy * 100, digits=2), "%")

# Show prediction probabilities for a few test samples
println("\nPrediction probabilities for first 5 test samples:")
for i in 1:5
    probs = test_predictions[i]
    pred_class = argmax(probs) - 1  # Convert to 0-9
    true_class = test_y[i]
    println("Sample $i: Predicted=$pred_class (prob=$(round(maximum(probs), digits=3))), True=$true_class")
end

In [ ]:
# Get predictions with uncertainty quantification
predictions_probit = predict(la, X[:, 1:100])  # First 100 training samples
predictions_probit_matrix = reduce(hcat, predictions_probit)
predictions_probit_reshaped = reshape(predictions_probit_matrix, 10, 100)  # 10 classes, 100 samples

# Calculate prediction uncertainty (entropy)
using Statistics
entropies = [-sum(p .* log.(p .+ 1e-8)) for p in predictions_probit]
println("Average prediction entropy: ", round(mean(entropies), digits=3))
println("Entropy std: ", round(std(entropies), digits=3))

In [ ]:
using DataFrames, CSV

# Create a DataFrame with MNIST class predictions
df = DataFrame()
for i in 0:9
    df[!, Symbol("class_$i")] = predictions_probit_reshaped[i+1, :]
end

# Add true labels and predicted labels
df[!, :true_label] = y[1:100]
predicted_labels = [argmax(predictions_probit[i]) - 1 for i in 1:100]
df[!, :predicted_label] = predicted_labels
df[!, :entropy] = entropies[1:100]

# Write table to CSV file
CSV.write("mnist_predictions_julia.csv", df)
println("Predictions saved to mnist_predictions_julia.csv")

In [ ]:
# Compare Laplace approximation vs plugin predictions
plugin_predictions = predict(la, X[:, 1:100]; link_approx=:plugin)
plugin_predictions_matrix = reduce(hcat, plugin_predictions)

# Calculate difference in uncertainty between methods
laplace_entropies = [-sum(p .* log.(p .+ 1e-8)) for p in predictions_probit]
plugin_entropies = [-sum(p .* log.(p .+ 1e-8)) for p in plugin_predictions]

println("Laplace method - Average entropy: ", round(mean(laplace_entropies), digits=3))
println("Plugin method - Average entropy: ", round(mean(plugin_entropies), digits=3))
println("Uncertainty difference: ", round(mean(laplace_entropies) - mean(plugin_entropies), digits=3))

# Show samples where methods disagree most
differences = abs.(laplace_entropies .- plugin_entropies)
top_diff_indices = sortperm(differences, rev=true)[1:5]

println("\nTop 5 samples with largest uncertainty differences:")
for (i, idx) in enumerate(top_diff_indices)
    laplace_pred = argmax(predictions_probit[idx]) - 1
    plugin_pred = argmax(plugin_predictions[idx]) - 1
    true_label = y[idx]
    println("Sample $idx: True=$true_label, Laplace=$laplace_pred, Plugin=$plugin_pred, Diff=$(round(differences[idx], digits=3))")
end

In [ ]:
# Visualize some MNIST digits with predictions
using Plots

# Function to plot MNIST digit
function plot_digit(digit_data, title_str)
    digit_img = reshape(digit_data, 28, 28)
    heatmap(digit_img, color=:grays, aspect_ratio=:equal, 
            title=title_str, showaxis=false, grid=false)
end

# Plot first 6 test samples with predictions
plt_list = []
for i in 1:6
    digit = test_x[:, i]
    true_label = test_y[i]
    pred_probs = predict(la, digit)
    pred_label = argmax(pred_probs[1]) - 1
    confidence = maximum(pred_probs[1])
    
    title_str = "True: $true_label, Pred: $pred_label\nConf: $(round(confidence, digits=2))"
    plt = plot_digit(digit, title_str)
    push!(plt_list, plt)
end

plot(plt_list..., layout=(2, 3), size=(600, 400))
savefig("mnist_predictions_sample.png")
println("Sample predictions plot saved as mnist_predictions_sample.png")

In [ ]:
# Final model analysis
println("=== MNIST Classification with Laplace Approximation ===")
println("Dataset: MNIST ($(n_samples) training samples)")
println("Model: 3-layer MLP with $(n_hidden) hidden units")
println("Classes: 10 (digits 0-9)")

# Training accuracy
train_preds = predict(la, X)
train_pred_classes = [argmax(p) - 1 for p in train_preds]
train_accuracy = mean(train_pred_classes .== y)
println("\nTraining Accuracy: $(round(train_accuracy * 100, digits=2))%")

# Test accuracy on subset
test_subset_size = 1000
test_preds = predict(la, test_x[:, 1:test_subset_size])
test_pred_classes = [argmax(p) - 1 for p in test_preds]
test_accuracy = mean(test_pred_classes .== test_y[1:test_subset_size])
println("Test Accuracy: $(round(test_accuracy * 100, digits=2))%")

# Uncertainty analysis
test_entropies = [-sum(p .* log.(p .+ 1e-8)) for p in test_preds]
correct_mask = test_pred_classes .== test_y[1:test_subset_size]
correct_entropies = test_entropies[correct_mask]
incorrect_entropies = test_entropies[.!correct_mask]

println("\nUncertainty Analysis:")
println("Average entropy (correct predictions): $(round(mean(correct_entropies), digits=3))")
println("Average entropy (incorrect predictions): $(round(mean(incorrect_entropies), digits=3))")
println("Entropy difference: $(round(mean(incorrect_entropies) - mean(correct_entropies), digits=3))")

# Class-wise accuracy
println("\nPer-class accuracy:")
for digit in 0:9
    mask = test_y[1:test_subset_size] .== digit
    if sum(mask) > 0
        digit_accuracy = mean(test_pred_classes[mask] .== digit)
        println("Digit $digit: $(round(digit_accuracy * 100, digits=1))% ($(sum(mask)) samples)")
    end
end
